# Phase 1 — CanViT Reproduction on ImageNette

**Run on Google Colab with GPU runtime.**

Goal: Verify that our sequential inference pipeline produces sensible results before implementing any policies.

Dataset: ImageNette (10-class ImageNet subset). **Not the official ImageNet-1k benchmark.**  
Results from this notebook are development-only and will be clearly labeled as such.

Checks:
- Sequential inference produces increasing confidence over glimpses
- Results are plausible (not obviously broken)
- Pipeline matches the experimental protocol structure

Creates: `docs/reproduction_report.md`

## 1.0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['HF_HOME'] = '/content/drive/MyDrive/canvit_cache'
RESULTS_DIR = '/content/drive/MyDrive/canvit_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Clone / update repo
REPO_URL = 'https://github.com/johnsaurabh/active-canvit-gaze.git'
REPO_DIR = '/content/active-canvit-gaze'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

!pip install -q 'canvit-pytorch @ git+https://github.com/m2b3/CanViT-PyTorch.git'
!pip install -q huggingface_hub scipy
print('Setup complete.')

## 1.1 — Download ImageNette (skip if already done)

In [ ]:
DATA_DIR = '/content/drive/MyDrive/data'
IMAGENETTE_DIR = os.path.join(DATA_DIR, 'imagenette2-320')

if not os.path.exists(IMAGENETTE_DIR):
    print('Downloading ImageNette (~1.4GB) directly to Drive...')
    !wget -q --show-progress https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz -P {DATA_DIR}
    print('Extracting...')
    !tar -xzf {DATA_DIR}/imagenette2-320.tgz -C {DATA_DIR}
    print('Done.')
else:
    print(f'ImageNette already exists at {IMAGENETTE_DIR}')

VAL_DIR = os.path.join(IMAGENETTE_DIR, 'val')
classes_on_disk = sorted(os.listdir(VAL_DIR))
print(f'Val classes: {classes_on_disk}')
print(f'Total val images: {sum(len(os.listdir(os.path.join(VAL_DIR, c))) for c in classes_on_disk)}')

## 1.2 — Load model

In [ ]:
import torch
from canvit_pytorch import CanViTForImageClassification, Viewpoint
from canvit_pytorch import sample_at_viewpoint
from canvit_pytorch.preprocess import preprocess

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CHECKPOINT = 'canvit/canvitb16-add-vpe-finetune-g128px-s512px-in1k-2026-04-06'
model = CanViTForImageClassification.from_pretrained(CHECKPOINT).eval().to(DEVICE)
print(f'Model loaded.')

CANVAS_GRID_SIZE = 32
GLIMPSE_SIZE_PX = 128
SCENE_SIZE = 512
LOCAL_SCALE = 0.25
GLIMPSE_BUDGETS = [0, 1, 2, 3, 4, 5, 6, 8]

## 1.3 — Build ImageNette val dataset

ImageNette uses WordNet IDs (e.g. `n01440764`). The finetuned CanViT was trained on ImageNet-1k which uses the same IDs, so label mapping works directly.

In [ ]:
import json
from pathlib import Path
from PIL import Image
import random

# ImageNet class index: maps synset -> class index (0-999)
# Download the standard ImageNet class index
import requests
HEADERS = {'User-Agent': 'active-canvit-gaze/1.0'}
idx_url = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
class_idx = json.loads(requests.get(idx_url, headers=HEADERS, timeout=10).text)
# class_idx: {'0': ['n01440764', 'tench'], '1': [...], ...}
synset_to_idx = {v[0]: int(k) for k, v in class_idx.items()}
idx_to_name  = {int(k): v[1] for k, v in class_idx.items()}
print(f'Loaded {len(synset_to_idx)} synset mappings.')

# Collect val samples
samples = []
for synset in sorted(os.listdir(VAL_DIR)):
    synset_dir = Path(VAL_DIR) / synset
    if not synset_dir.is_dir():
        continue
    label = synset_to_idx.get(synset, -1)
    for img_path in sorted(synset_dir.iterdir()):
        if img_path.suffix.lower() in {'.jpeg', '.jpg', '.png'}:
            samples.append({'path': str(img_path), 'synset': synset, 'label': label})

print(f'Total val samples: {len(samples)}')
print(f'Classes found: {set(s["synset"] for s in samples)}')

# Fixed reproducible subset — 100 images for development
rng = random.Random(42)
subset = rng.sample(samples, min(100, len(samples)))
print(f'\nUsing fixed subset of {len(subset)} images (seed=42).')

## 1.4 — Run sequential inference on subset

Policy: random (seeded) — we are not testing policies yet, just verifying the pipeline.

For each image: T=0 full scene, then T=1..8 random local glimpses.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

transform = preprocess(SCENE_SIZE)

def make_viewpoint(x, y, s):
    return Viewpoint(
        centers=torch.tensor([[x, y]], device=DEVICE),
        scales=torch.tensor([s], device=DEVICE),
    )

def run_random_sequence(image_tensor, n_local=8, seed=0):
    """Run full-scene + n_local random glimpses. Returns logits at each step."""
    rng = random.Random(seed)
    max_center = 1.0 - LOCAL_SCALE

    state = model.init_state(batch_size=1, canvas_grid_size=CANVAS_GRID_SIZE)
    all_logits = []

    with torch.inference_mode():
        # T=0: full scene
        vp = make_viewpoint(0., 0., 1.0)
        glimpse = sample_at_viewpoint(spatial=image_tensor, viewpoint=vp, glimpse_size_px=GLIMPSE_SIZE_PX)
        logits, state = model(glimpse=glimpse, state=state, viewpoint=vp)
        all_logits.append(logits.cpu())

        # T=1..n_local: random local glimpses
        for _ in range(n_local):
            x = rng.uniform(-max_center, max_center)
            y = rng.uniform(-max_center, max_center)
            vp = make_viewpoint(x, y, LOCAL_SCALE)
            glimpse = sample_at_viewpoint(spatial=image_tensor, viewpoint=vp, glimpse_size_px=GLIMPSE_SIZE_PX)
            logits, state = model(glimpse=glimpse, state=state, viewpoint=vp)
            all_logits.append(logits.cpu())

    return all_logits  # list of T+1 tensors, each [1, 1000]

# Run on subset
records = []
N_LOCAL = max(GLIMPSE_BUDGETS)  # 8

for sample in tqdm(subset, desc='Running inference'):
    try:
        img = Image.open(sample['path']).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(DEVICE)
        all_logits = run_random_sequence(img_tensor, n_local=N_LOCAL, seed=0)

        for t, logits in enumerate(all_logits):
            probs = torch.softmax(logits, dim=-1)
            pred = int(probs.argmax())
            conf = float(probs.max())
            correct_top1 = int(pred == sample['label'])
            top5 = torch.topk(probs, k=5, dim=-1).indices[0].tolist()
            correct_top5 = int(sample['label'] in top5)

            records.append({
                'image_id': sample['path'],
                'synset': sample['synset'],
                'true_label': sample['label'],
                'timestep': t,
                'pred_label': pred,
                'pred_name': idx_to_name.get(pred, str(pred)),
                'confidence': conf,
                'correct_top1': correct_top1,
                'correct_top5': correct_top5,
            })
    except Exception as e:
        print(f'Failed on {sample["path"]}: {e}')

df = pd.DataFrame(records)
print(f'\nDone. {len(subset)} images × {N_LOCAL+1} timesteps = {len(df)} records.')

## 1.5 — Results

In [ ]:
import matplotlib.pyplot as plt

# Accuracy at each glimpse budget
acc_by_t = df.groupby('timestep')['correct_top1'].mean()
acc5_by_t = df.groupby('timestep')['correct_top5'].mean()
conf_by_t = df.groupby('timestep')['confidence'].mean()

print('Timestep | Top-1 Acc | Top-5 Acc | Mean Conf')
print('-' * 50)
for t in sorted(acc_by_t.index):
    if t in GLIMPSE_BUDGETS:
        print(f'  T={t:2d}    |   {acc_by_t[t]:.3f}   |   {acc5_by_t[t]:.3f}   |   {conf_by_t[t]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

budgets = [t for t in GLIMPSE_BUDGETS if t in acc_by_t.index]
axes[0].plot(budgets, [acc_by_t[t] for t in budgets], 'o-', label='Top-1')
axes[0].plot(budgets, [acc5_by_t[t] for t in budgets], 's--', label='Top-5')
axes[0].set_xlabel('Glimpse budget')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Glimpses (random policy, ImageNette dev)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(budgets, [conf_by_t[t] for t in budgets], 'o-', color='orange')
axes[1].set_xlabel('Glimpse budget')
axes[1].set_ylabel('Mean confidence')
axes[1].set_title('Confidence vs Glimpses')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'phase1_random_baseline.png')
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')

In [ ]:
# Save raw results
results_path = os.path.join(RESULTS_DIR, 'phase1_raw.parquet')
df.to_parquet(results_path, index=False)
print(f'Raw results saved to {results_path}')

## 1.6 — Reproduction verdict

In [ ]:
acc_t0 = float(acc_by_t[0])
acc_t8 = float(acc_by_t[8]) if 8 in acc_by_t.index else float(acc_by_t.iloc[-1])
improves = acc_t8 >= acc_t0

print('=' * 50)
print('PHASE 1 REPRODUCTION REPORT')
print('=' * 50)
print(f'Dataset:         ImageNette val (100-image subset, seed=42)')
print(f'NOTE:            NOT official ImageNet-1k. Development only.')
print(f'Policy:          Random (seed=0)')
print(f'Glimpse scale:   {LOCAL_SCALE}')
print(f'Checkpoint:      {CHECKPOINT}')
print()
print(f'Top-1 at T=0:    {acc_t0:.3f}')
print(f'Top-1 at T=8:    {acc_t8:.3f}')
print(f'Improves with glimpses: {improves}')
print()

# Sanity checks
checks = [
    ('Accuracy > 0 at T=0', acc_t0 > 0.0),
    ('Accuracy plausible (> 0.05)', acc_t0 > 0.05),
    ('No NaN in results', df['correct_top1'].notna().all()),
    ('All 100 images processed', df['image_id'].nunique() == len(subset)),
]

all_pass = True
for name, result in checks:
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_pass = False
    print(f'  {status}: {name}')

print()
verdict = 'PASS' if all_pass else 'FAIL'
print(f'VERDICT: {verdict}')
print()
if verdict == 'PASS':
    print('Pipeline verified. Proceed to Phase 2 — policy comparison.')
else:
    print('Fix failing checks before proceeding.')